# Silver - physical_vendas_caixa

Desenvolvido por: Ygor Moraes

Este notebook lê a Bronze `physical_vendas_caixa` e grava a Silver tratada em Delta.

Regras aplicadas:
- manter `id_transacao` como string, pois é UUID;
- deduplicar vendas por `id_transacao`;
- converter `id_loja`, `id_caixa` e `id_operador` para integer;
- converter `dt_venda` para timestamp;
- converter `valor_total_venda` para decimal(10,2);
- manter apenas vendas com valor maior que zero;
- padronizar `cpf_cliente` mantendo apenas números;
- validar `tipo_pagamento` como Cartão, Dinheiro ou Pix;
- adicionar `silver_processed_at`;
- manter particionamento por `ano` e `mes`.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Silver de vendas de caixa.

from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    upper,
    regexp_replace,
    to_timestamp,
    current_timestamp,
    row_number
)

from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

BRONZE_TABLE = "physical_vendas_caixa"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

SILVER_TABLE = "physical_vendas_caixa"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"
SILVER_LOJAS_TABLE = "physical_lojas"
SILVER_LOJAS_PATH = f"{SILVER_BASE_PATH}{SILVER_LOJAS_TABLE}"

KEY_COLUMNS = ["id_transacao"]

LOJAS_REQUIRED_COLUMNS = [
    "id_loja"
]

SILVER_WRITE_MODE = "overwrite"

TIPOS_PAGAMENTO_VALIDOS = [
    "Cartão",
    "Dinheiro",
    "Pix"
]

BRONZE_REQUIRED_COLUMNS = [
    "id_transacao",
    "id_loja",
    "id_caixa",
    "id_operador",
    "dt_venda",
    "valor_total_venda",
    "cpf_cliente",
    "tipo_pagamento",
    "bronze_ingested_at",
    "bronze_source_file",
    "bronze_record_hash",
    "ano",
    "mes"
]

SILVER_REQUIRED_COLUMNS = [
    "id_transacao",
    "id_loja",
    "id_caixa",
    "id_operador",
    "dt_venda",
    "valor_total_venda",
    "cpf_cliente",
    "tipo_pagamento",
    "bronze_ingested_at",
    "bronze_source_file",
    "silver_processed_at",
    "ano",
    "mes"
]

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Bronze: {BRONZE_PATH}")
print(f"Destino Silver: {SILVER_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Bronze e valida se as colunas necessárias existem.

df_bronze = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

bronze_columns = df_bronze.columns

missing_bronze_columns = [
    c for c in BRONZE_REQUIRED_COLUMNS
    if c not in bronze_columns
]

if missing_bronze_columns:
    raise Exception(f"Colunas obrigatórias ausentes na Bronze: {missing_bronze_columns}")

total_bronze = df_bronze.count()

print("Bronze lida com sucesso.")
print(f"Total de registros na Bronze: {total_bronze}")

df_bronze.printSchema()

display(df_bronze.limit(10))

In [0]:
# Valida chave e observa campos críticos antes das transformações.

total_registros = df_bronze.count()

ids_nulos = df_bronze.filter(col("id_transacao").isNull()).count()

ids_duplicados = (
    df_bronze
    .groupBy("id_transacao")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Total de registros: {total_registros}")
print(f"IDs nulos: {ids_nulos}")
print(f"IDs duplicados: {ids_duplicados}")

display(
    df_bronze
    .select(
        "id_transacao",
        "id_loja",
        "id_caixa",
        "id_operador",
        "dt_venda",
        "valor_total_venda",
        "cpf_cliente",
        "tipo_pagamento"
    )
    .limit(10)
)

In [0]:
# Aplica limpeza, padronização e tipagem dos campos da Silver.

df_silver_base = (
    df_bronze
    .select(
        trim(col("id_transacao")).alias("id_transacao"),

        col("id_loja").cast("int").alias("id_loja"),
        col("id_caixa").cast("int").alias("id_caixa"),
        col("id_operador").cast("int").alias("id_operador"),

        to_timestamp(trim(col("dt_venda"))).alias("dt_venda"),

        regexp_replace(trim(col("valor_total_venda")), ",", ".")
        .cast(DecimalType(10, 2))
        .alias("valor_total_venda"),

        regexp_replace(trim(col("cpf_cliente")), "[^0-9]", "").alias("cpf_cliente"),

        when(upper(trim(col("tipo_pagamento"))).isin("CARTÃO", "CARTAO"), "Cartão")
        .when(upper(trim(col("tipo_pagamento"))) == "DINHEIRO", "Dinheiro")
        .when(upper(trim(col("tipo_pagamento"))) == "PIX", "Pix")
        .otherwise(None)
        .alias("tipo_pagamento"),

        col("bronze_ingested_at"),
        col("bronze_source_file"),
        current_timestamp().alias("silver_processed_at"),
        col("ano"),
        col("mes")
    )
)

display(df_silver_base.limit(10))

In [0]:
# Verifica se as conversões principais geraram nulos ou valores inválidos.

id_transacao_nulo = df_silver_base.filter(col("id_transacao").isNull()).count()
id_loja_nulo = df_silver_base.filter(col("id_loja").isNull()).count()
id_caixa_nulo = df_silver_base.filter(col("id_caixa").isNull()).count()
id_operador_nulo = df_silver_base.filter(col("id_operador").isNull()).count()
dt_venda_nula = df_silver_base.filter(col("dt_venda").isNull()).count()
valor_nulo = df_silver_base.filter(col("valor_total_venda").isNull()).count()
valor_invalido = df_silver_base.filter(col("valor_total_venda") <= 0).count()
tipo_pagamento_nulo = df_silver_base.filter(col("tipo_pagamento").isNull()).count()

print(f"id_transacao nulo: {id_transacao_nulo}")
print(f"id_loja nulo após cast: {id_loja_nulo}")
print(f"id_caixa nulo após cast: {id_caixa_nulo}")
print(f"id_operador nulo após cast: {id_operador_nulo}")
print(f"dt_venda nula após conversão: {dt_venda_nula}")
print(f"valor_total_venda nulo após cast: {valor_nulo}")
print(f"valor_total_venda <= 0: {valor_invalido}")
print(f"tipo_pagamento inválido/nulo: {tipo_pagamento_nulo}")

display(
    df_silver_base
    .groupBy("tipo_pagamento")
    .count()
    .orderBy("tipo_pagamento")
)

In [0]:
# Valida se id_loja das vendas existe na Silver de physical_lojas.

df_lojas = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_LOJAS_PATH)
)

validate_required_columns(df_lojas, LOJAS_REQUIRED_COLUMNS)

df_lojas_validas = (
    df_lojas
    .select(
        col("id_loja").cast("int").alias("id_loja")
    )
    .filter(col("id_loja").isNotNull())
    .dropDuplicates(["id_loja"])
)

lojas_invalidas = (
    df_silver_base
    .select("id_loja")
    .dropDuplicates(["id_loja"])
    .join(
        df_lojas_validas,
        on="id_loja",
        how="left_anti"
    )
)

qtd_lojas_validas = df_lojas_validas.count()
qtd_lojas_invalidas = lojas_invalidas.count()

print(f"Total de lojas válidas na Silver physical_lojas: {qtd_lojas_validas}")
print(f"Total de id_loja das vendas sem match em physical_lojas: {qtd_lojas_invalidas}")

if qtd_lojas_invalidas > 0:
    print("Amostra de id_loja sem match:")
    display(lojas_invalidas.orderBy("id_loja"))

    raise Exception("Erro: existem vendas com id_loja inexistente em physical_lojas.")

print("Validação OK: todos os id_loja das vendas existem em physical_lojas.")

In [0]:
# Mantém apenas vendas com campos essenciais válidos.

df_silver_validado = (
    df_silver_base
    .filter(col("id_transacao").isNotNull())
    .filter(col("id_loja").isNotNull())
    .filter(col("id_caixa").isNotNull())
    .filter(col("id_operador").isNotNull())
    .filter(col("dt_venda").isNotNull())
    .filter(col("valor_total_venda") > 0)
    .filter(col("tipo_pagamento").isNotNull())
)

total_validado = df_silver_validado.count()

print(f"Total antes das regras: {total_registros}")
print(f"Total após regras críticas: {total_validado}")
print(f"Registros removidos: {total_registros - total_validado}")

In [0]:
# Deduplica vendas pela chave principal.

window_vendas = (
    Window
    .partitionBy("id_transacao")
    .orderBy(col("bronze_ingested_at").desc())
)

df_silver = (
    df_silver_validado
    .withColumn("row_number", row_number().over(window_vendas))
    .filter(col("row_number") == 1)
    .drop("row_number")
)

total_silver = df_silver.count()

duplicados_silver = (
    df_silver
    .groupBy("id_transacao")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Total Silver final: {total_silver}")
print(f"IDs duplicados na Silver: {duplicados_silver}")

display(df_silver.limit(10))

In [0]:
# Valida se a Silver possui as colunas esperadas.

silver_columns = df_silver.columns

missing_silver_columns = [
    c for c in SILVER_REQUIRED_COLUMNS
    if c not in silver_columns
]

if missing_silver_columns:
    raise Exception(f"Colunas obrigatórias ausentes na Silver: {missing_silver_columns}")

print("Schema final da Silver validado.")

df_silver.printSchema()

In [0]:
# Grava a Silver de vendas de caixa em Delta.

(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print("Silver gravada com sucesso.")
print(f"Path: {SILVER_PATH}")

In [0]:
# Lê a Silver gravada e valida volume e chave.

df_silver_gravada = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

total_gravado = df_silver_gravada.count()

duplicados_gravados = (
    df_silver_gravada
    .groupBy("id_transacao")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Total Silver gravada: {total_gravado}")
print(f"IDs duplicados na Silver gravada: {duplicados_gravados}")

display(df_silver_gravada.limit(10))